In [2]:
import sqlite3
import pandas as pd

# ==============================
# 1. KHỞI TẠO DATABASE
# ==============================
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# Tạo bảng
cursor.execute("""
CREATE TABLE student (
    student_id INTEGER,
    name TEXT,
    class TEXT,
    course_id INTEGER,
    score REAL
)
""")

cursor.execute("""
CREATE TABLE course (
    id INTEGER,
    course_name TEXT
)
""")

# Insert dữ liệu
students = [
(1,"Nguyen Minh Hoang","May Tinh",12,6.7),
(2,"Tran Thi Lan","Kinh Te",34,9.2),
(3,"Pham Van Nam","Toan Tin",None,7.9),
(4,"Le Thanh Huyen","Toan Tin",20,7.2),
(5,"Vu Quoc Anh","May Tinh",24,8.0),
(6,"Dang Thuy Linh","May Tinh",24,5.5),
(7,"Bui Tien Dung","Kinh Te",34,9.2),
(8,"Ho Ngoc Mai","Toan Tin",20,8.8),
(9,"Duong Huu Phuc","Kinh Te",None,7.2),
(10,"Cao Thi Hanh","May Tinh",None,7.0)
]

courses = [
(12,"Giai tich"),
(34,"Thong ke"),
(26,"Tin hoc")
]

cursor.executemany("INSERT INTO student VALUES (?,?,?,?,?)", students)
cursor.executemany("INSERT INTO course VALUES (?,?)", courses)
conn.commit()

# ==============================
# 2. JOIN
# ==============================
print("\n--- DESCARTES ---")
print(pd.read_sql("SELECT * FROM student, course", conn))

print("\n--- INNER JOIN ---")
print(pd.read_sql("""
SELECT * FROM student s
INNER JOIN course c ON s.course_id = c.id
""", conn))

print("\n--- LEFT JOIN ---")
print(pd.read_sql("""
SELECT * FROM student s
LEFT JOIN course c ON s.course_id = c.id
""", conn))

print("\n--- RIGHT JOIN (giả lập) ---")
print(pd.read_sql("""
SELECT * FROM course c
LEFT JOIN student s ON s.course_id = c.id
""", conn))

print("\n--- FULL OUTER JOIN (giả lập) ---")
print(pd.read_sql("""
SELECT * FROM student s
LEFT JOIN course c ON s.course_id = c.id
UNION
SELECT * FROM course c
LEFT JOIN student s ON s.course_id = c.id
""", conn))

# ==============================
# 3. LÀM SẠCH DỮ LIỆU
# ==============================
cursor.execute("""
DELETE FROM student
WHERE course_id NOT IN (SELECT id FROM course)
""")
conn.commit()

# ==============================
# 3a. THỐNG KÊ THEO LỚP
# ==============================
print("\n--- THỐNG KÊ THEO LỚP ---")
print(pd.read_sql("""
SELECT class,
       COUNT(*) AS total_students,
       ROUND(AVG(score),2) AS avg_score
FROM student
GROUP BY class
""", conn))

# ==============================
# 3b. THEO MÔN
# ==============================
print("\n--- THỐNG KÊ THEO MÔN ---")
print(pd.read_sql("""
SELECT c.course_name,
       COUNT(*) AS total_students,
       ROUND(AVG(score),2) AS avg_score
FROM student s
JOIN course c ON s.course_id = c.id
GROUP BY c.course_name
""", conn))

# ==============================
# 3c. PHÂN LOẠI
# ==============================
print("\n--- PHÂN LOẠI THI ĐUA ---")
print(pd.read_sql("""
SELECT c.course_name,
       ROUND(AVG(score),2) AS avg_score,
       CASE
           WHEN AVG(score) >= 9 THEN 'Xuat sac'
           WHEN AVG(score) >= 6 THEN 'Tot'
           ELSE 'Kem'
       END AS classification
FROM student s
JOIN course c ON s.course_id = c.id
GROUP BY c.course_name
""", conn))

# ==============================
# 4. XẾP HẠNG
# ==============================
print("\n--- XẾP HẠNG TOÀN BỘ ---")
print(pd.read_sql("""
SELECT *,
RANK() OVER (ORDER BY score DESC) AS rank_score
FROM student
""", conn))

print("\n--- TOP 3 CAO NHẤT ---")
print(pd.read_sql("""
SELECT * FROM student ORDER BY score DESC LIMIT 3
""", conn))

print("\n--- TOP 3 THẤP NHẤT ---")
print(pd.read_sql("""
SELECT * FROM student ORDER BY score ASC LIMIT 3
""", conn))

print("\n--- XẾP HẠNG THEO LỚP ---")
print(pd.read_sql("""
SELECT *,
RANK() OVER (PARTITION BY class ORDER BY score DESC) AS rank_class
FROM student
""", conn))

print("\n--- XẾP HẠNG THEO MÔN ---")
print(pd.read_sql("""
SELECT *,
RANK() OVER (PARTITION BY course_id ORDER BY score DESC) AS rank_course
FROM student
""", conn))

# ==============================
# 5. GRADUATION DATE
# ==============================
cursor.execute("ALTER TABLE student ADD COLUMN graduation_date DATETIME")

cursor.execute("""
UPDATE student
SET graduation_date = datetime('now', '+' || CAST(score AS INTEGER) || ' days')
""")

conn.commit()

print("\n--- STUDENT SAU KHI THÊM GRADUATION DATE ---")
print(pd.read_sql("SELECT * FROM student", conn))


--- DESCARTES ---
    student_id               name     class  course_id  score  id course_name
0            1  Nguyen Minh Hoang  May Tinh       12.0    6.7  12   Giai tich
1            1  Nguyen Minh Hoang  May Tinh       12.0    6.7  34    Thong ke
2            1  Nguyen Minh Hoang  May Tinh       12.0    6.7  26     Tin hoc
3            2       Tran Thi Lan   Kinh Te       34.0    9.2  12   Giai tich
4            2       Tran Thi Lan   Kinh Te       34.0    9.2  34    Thong ke
5            2       Tran Thi Lan   Kinh Te       34.0    9.2  26     Tin hoc
6            3       Pham Van Nam  Toan Tin        NaN    7.9  12   Giai tich
7            3       Pham Van Nam  Toan Tin        NaN    7.9  34    Thong ke
8            3       Pham Van Nam  Toan Tin        NaN    7.9  26     Tin hoc
9            4     Le Thanh Huyen  Toan Tin       20.0    7.2  12   Giai tich
10           4     Le Thanh Huyen  Toan Tin       20.0    7.2  34    Thong ke
11           4     Le Thanh Huyen  Toan Tin  